# HAA Leader Sector System v2.1 — GitHub Sync & Execution Check

이 runner는 GitHub 최신 v2.1을 내려받아 KRX preflight와 자체 테스트까지만 실행합니다. `run_daily()`는 호출하지 않습니다.


In [ ]:
REPOSITORY = "hyunsungkim73/HAA_Leader_Sector_System"
BRANCH = "codex/haa-v2.1-hardening"  # PR 병합 후 main으로 변경
SOURCE_NOTEBOOK = "HAA_Leader_Sector_System_v2_1.ipynb"


In [ ]:
import os
from google.colab import userdata

for secret_name in ("KRX_ID", "KRX_PW"):
    try:
        secret_value = userdata.get(secret_name)
    except Exception as e:
        raise RuntimeError(
            f"Colab Secret {secret_name!r} is missing or access is disabled."
        ) from e

    if not str(secret_value).strip():
        raise RuntimeError(f"Colab Secret {secret_name!r} is empty.")

    os.environ[secret_name] = str(secret_value).strip()

if "@" in os.environ["KRX_ID"]:
    raise RuntimeError(
        "KRX_ID must be the KRX data-portal member ID, not an email address."
    )

print("Colab Secrets loaded (values are not displayed).")


In [ ]:
import base64
import pathlib
import requests

api_url = (
    f"https://api.github.com/repos/{REPOSITORY}/contents/{SOURCE_NOTEBOOK}"
)
response = requests.get(
    api_url,
    params={"ref": BRANCH},
    headers={
        "Accept": "application/vnd.github+json",
        "Cache-Control": "no-cache",
    },
    timeout=30,
)
response.raise_for_status()
payload = response.json()

source_path = pathlib.Path("/content") / SOURCE_NOTEBOOK
source_path.write_bytes(base64.b64decode(payload["content"]))

SOURCE_COMMIT_BLOB_SHA = payload["sha"]
print(f"SYNC_OK branch={BRANCH} blob_sha={SOURCE_COMMIT_BLOB_SHA}")
print(f"Downloaded: {source_path}")


In [ ]:
import json
import subprocess
import sys

executed_path = pathlib.Path("/content/HAA_Leader_Sector_System_v2_1_executed.ipynb")
run_env = os.environ.copy()
run_env["HAA_EXECUTION_CHECK_ONLY"] = "1"
run_env["HAA_AUTO_RUN_KRX_PREFLIGHT"] = "1"

command = [
    sys.executable,
    "-m",
    "jupyter",
    "nbconvert",
    "--to", "notebook",
    "--execute",
    "--ExecutePreprocessor.timeout=2400",
    "--output", str(executed_path),
    str(source_path),
]

completed = subprocess.run(
    command,
    env=run_env,
    text=True,
    capture_output=True,
    timeout=2700,
)

if completed.returncode != 0:
    print(completed.stdout[-4000:])
    print(completed.stderr[-4000:])
    raise RuntimeError(
        f"Notebook execution failed with exit code {completed.returncode}."
    )

executed = json.loads(executed_path.read_text(encoding="utf-8"))
errors = []
output_text = []

for cell_index, cell in enumerate(executed.get("cells", [])):
    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            errors.append({
                "cell": cell_index,
                "ename": output.get("ename"),
                "evalue": output.get("evalue"),
            })
        text_value = output.get("text", "")
        if isinstance(text_value, list):
            text_value = "".join(text_value)
        if text_value:
            output_text.append(str(text_value))

if errors:
    raise RuntimeError(f"Executed notebook contains errors: {errors}")

combined_output = "\n".join(output_text)
if "AUTO_KRX_PREFLIGHT_OK" not in combined_output:
    raise RuntimeError("KRX preflight success marker was not produced.")

smoke_lines = [
    line.strip()
    for line in combined_output.splitlines()
    if "MARKET_SMOKE_TEST_OK" in line
]
if not smoke_lines:
    raise RuntimeError("Market smoke-test success marker was not produced.")
if "SELF_TEST_OK" not in combined_output and "'status': 'ok'" not in combined_output:
    # The self-test cell's display representation differs by kernel version.
    self_test_cell = next(
        (c for c in executed["cells"] if c.get("id") == "v21-self-tests"),
        None,
    )
    if not self_test_cell or not self_test_cell.get("outputs"):
        raise RuntimeError("v2.1 self-test output was not produced.")

print(smoke_lines[-1])
print("EXECUTION_CHECK_OK")
print(f"branch={BRANCH}")
print(f"source_blob_sha={SOURCE_COMMIT_BLOB_SHA}")
print(f"executed_notebook={executed_path}")
print("run_daily_called=False")
